# ERA5 Visualisation over India (EarthKit) — Fixed Version

**Features**: EarthKit (CDS) → NetCDF, correct units, India bbox, Cartopy maps, ipywidgets animation, GIF export.

> Run the sanity checks if numbers look off.

In [ ]:

# If running first time in a fresh environment, uncomment:
# %pip -q install earthkit-data xarray netCDF4 cartopy ipywidgets matplotlib Pillow


In [ ]:

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import ipywidgets as widgets
from IPython.display import display, Markdown

import earthkit.data as ekd


## Config

In [ ]:

# India bounding box
INDIA_BBOX = {"north": 37.5, "west": 68.0, "south": 6.0, "east": 97.5}

# Date/time selection
DATE = "2025-08-01"   # YYYY-MM-DD
HOURS = [f"{h:02d}:00" for h in range(0, 24, 3)]  # 0,3,6,...,21 UTC

# Variables
VARS_SINGLE = ["2m_temperature", "mean_sea_level_pressure"]

# Output
OUTDIR = Path("./era5_outputs"); OUTDIR.mkdir(exist_ok=True, parents=True)

# Optional pressure level fetch
FETCH_PL_850 = False


## Fetch ERA5 Single Levels via EarthKit (NetCDF)

In [ ]:

def fetch_era5_single_as_netcdf(date, hours, bbox, vars_single, outdir=OUTDIR):
    request = {
        "product_type": "reanalysis",
        "variable": vars_single,
        "year": str(datetime.fromisoformat(date).year),
        "month": f"{datetime.fromisoformat(date).month:02d}",
        "day": f"{datetime.fromisoformat(date).day:02d}",
        "time": hours,
        "area": [bbox["north"], bbox["west"], bbox["south"], bbox["east"]],
        "format": "netcdf",
        "grid": [0.25, 0.25],
    }
    ds = ekd.from_source("cds", "reanalysis-era5-single-levels", request)
    out_nc = outdir / f"era5_single_{date.replace('-', '')}.nc"
    try:
        return Path(ekd.to_local(ds, path=str(out_nc)))
    finally:
        try: ds.close()
        except Exception: pass

single_nc = fetch_era5_single_as_netcdf(DATE, HOURS, INDIA_BBOX, VARS_SINGLE, OUTDIR)
single_nc


## Open, subset, and fix units

In [ ]:

ds_single = xr.open_dataset(single_nc)

ds_ind = ds_single.sel(
    latitude=slice(INDIA_BBOX["north"], INDIA_BBOX["south"]),
    longitude=slice(INDIA_BBOX["west"], INDIA_BBOX["east"]),
)

assert "t2m" in ds_ind.variables, "t2m not found; ensure '2m_temperature' was downloaded."
assert "msl" in ds_ind.variables, "msl not found; ensure 'mean_sea_level_pressure' was downloaded."

t2m_c = ds_ind["t2m"] - 273.15     # K -> °C
msl_hpa = ds_ind["msl"] / 100.0    # Pa -> hPa

def sanity_report(da, name, unit, exp_min=None, exp_max=None):
    vmin = float(da.min().values)
    vmax = float(da.max().values)
    msg = f"{name}: min={vmin:.2f} {unit}, max={vmax:.2f} {unit}"
    if exp_min is not None and exp_max is not None:
        msg += f" | expected roughly [{exp_min}, {exp_max}] over India"
    print(msg)

sanity_report(t2m_c, "2m temperature", "°C", exp_min=-10, exp_max=50)
sanity_report(msl_hpa, "MSLP", "hPa", exp_min=950, exp_max=1050)

ds_ind


## Mapping utilities (Cartopy)

In [ ]:

def plot_map(da2d, *, title="", units="", vmin=None, vmax=None):
    fig = plt.figure(figsize=(8, 6), dpi=120)
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([INDIA_BBOX["west"], INDIA_BBOX["east"], INDIA_BBOX["south"], INDIA_BBOX["north"]], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAKES, edgecolor='black', facecolor='none', linewidth=0.3)
    ax.add_feature(cfeature.RIVERS, edgecolor='black', linewidth=0.3)

    gl = ax.gridlines(draw_labels=True, linewidth=0.25, x_inline=False, y_inline=False)
    gl.top_labels = False
    gl.right_labels = False

    img = da2d.plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, vmin=vmin, vmax=vmax)
    cb = plt.colorbar(img, ax=ax, shrink=0.7, pad=0.02)
    cb.set_label(units)

    ax.set_title(title)
    plt.tight_layout()
    plt.show()


## Interactive: 2m Temperature (°C) with Play/Pause

In [ ]:

da = t2m_c
times = ds_ind["time"].values
nt = len(times)

play = widgets.Play(interval=400, value=0, min=0, max=nt-1, step=1, description="Play")
slider = widgets.IntSlider(value=0, min=0, max=nt-1, step=1, description="t")
widgets.jslink((play, 'value'), (slider, 'value'))

@widgets.interact(i=slider)
def _view(i=0):
    ts = np.datetime_as_string(times[i], unit='m')
    plot_map(da.isel(time=i), title=f"ERA5 2m Temperature (°C) — {ts}", units="°C", vmin=10, vmax=45)

display(play)


## Interactive: Mean Sea-Level Pressure (hPa) with Play/Pause

In [ ]:

da_p = msl_hpa
times = ds_ind["time"].values
nt = len(times)

play_p = widgets.Play(interval=400, value=0, min=0, max=nt-1, step=1, description="Play")
slider_p = widgets.IntSlider(value=0, min=0, max=nt-1, step=1, description="t")
widgets.jslink((play_p, 'value'), (slider_p, 'value'))

@widgets.interact(i=slider_p)
def _view_p(i=0):
    ts = np.datetime_as_string(times[i], unit='m')
    plot_map(da_p.isel(time=i), title=f"ERA5 MSLP (hPa) — {ts}", units="hPa", vmin=980, vmax=1025)

display(play_p)


## (Optional) ERA5 Pressure Level: 850 hPa Temperature

In [ ]:

if FETCH_PL_850:
    def fetch_era5_pl850_as_netcdf(date, hours, bbox, outdir=OUTDIR):
        request = {
            "product_type": "reanalysis",
            "variable": ["temperature"],
            "pressure_level": ["850"],
            "year": str(datetime.fromisoformat(date).year),
            "month": f"{datetime.fromisoformat(date).month:02d}",
            "day": f"{datetime.fromisoformat(date).day:02d}",
            "time": hours,
            "area": [bbox["north"], bbox["west"], bbox["south"], bbox["east"]],
            "format": "netcdf",
            "grid": [0.25, 0.25],
        }
        ds = ekd.from_source("cds", "reanalysis-era5-pressure-levels", request)
        out_nc = outdir / f"era5_pl850_{date.replace('-', '')}.nc"
        try:
            return Path(ekd.to_local(ds, path=str(out_nc)))
        finally:
            try: ds.close()
            except Exception: pass

    pl_nc = fetch_era5_pl850_as_netcdf(DATE, HOURS, INDIA_BBOX, OUTDIR)
    ds_pl = xr.open_dataset(pl_nc).sel(
        latitude=slice(INDIA_BBOX["north"], INDIA_BBOX["south"]),
        longitude=slice(INDIA_BBOX["west"], INDIA_BBOX["east"]),
    )
    assert "t" in ds_pl.variables, "850-hPa temperature variable 't' not found."
    t850_c = ds_pl["t"] - 273.15

    times_pl = ds_pl["time"].values
    nt_pl = len(times_pl)

    play_pl = widgets.Play(interval=400, value=0, min=0, max=nt_pl-1, step=1, description="Play")
    slider_pl = widgets.IntSlider(value=0, min=0, max=nt_pl-1, step=1, description="t")
    widgets.jslink((play_pl, 'value'), (slider_pl, 'value'))

    @widgets.interact(i=slider_pl)
    def _view_pl(i=0):
        ts = np.datetime_as_string(times_pl[i], unit='m')
        plot_map(t850_c.isel(time=i), title=f"ERA5 850 hPa Temperature (°C) — {ts}", units="°C", vmin=-10, vmax=25)

    display(play_pl)
else:
    display(Markdown("> To fetch 850 hPa temperature, set `FETCH_PL_850 = True` above and rerun."))


## Export GIF (2m temperature)

In [ ]:

from matplotlib.animation import FuncAnimation, PillowWriter

gif_path = OUTDIR / "era5_t2m_india.gif"

vmin = float(t2m_c.min().values)
vmax = float(t2m_c.max().values)

fig = plt.figure(figsize=(8, 6), dpi=120)
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([INDIA_BBOX["west"], INDIA_BBOX["east"], INDIA_BBOX["south"], INDIA_BBOX["north"]], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.LAKES, edgecolor='black', facecolor='none', linewidth=0.3)
ax.add_feature(cfeature.RIVERS, edgecolor='black', linewidth=0.3)
gl = ax.gridlines(draw_labels=True, linewidth=0.25, x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False

im = t2m_c.isel(time=0).plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, vmin=vmin, vmax=vmax)
cb = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cb.set_label("°C")
title = ax.set_title("")

def update(i):
    ts = np.datetime_as_string(ds_ind["time"].values[i], unit='m')
    # For pcolormesh-type artists, set_array expects a flattened array; safest is to redraw
    for coll in im.collections: coll.remove()
    new_im = t2m_c.isel(time=i).plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, vmin=vmin, vmax=vmax)
    return [new_im]

anim = FuncAnimation(fig, update, frames=len(ds_ind["time"]), interval=400, blit=False)
anim.save(gif_path, writer=PillowWriter(fps=3))
plt.close(fig)

gif_path


### GIF Preview (Markdown)
![ERA5 2m temperature over India](era5_outputs/era5_t2m_india.gif)

## Notes
- ERA5 `t2m` is Kelvin → converted to °C.
- ERA5 `msl` is Pa → converted to hPa.
- If widgets don't show, ensure `ipywidgets>=8`.
- If Cartopy errors, update `cartopy` and `shapely`.